### 1. Imports

In [1]:
import os
import requests
from typing import TypedDict

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END

### 2. Configurations

In [2]:
with open(r"E:\Lenovo Ideapad 330\company-material\digital-workforce-transformation\ai-upskill-11\key-vault\openai\api.key") as f:
    openai_api_key = f.read().strip()
os.environ["OPENAI_API_KEY"] = openai_api_key

In [3]:
CIS_URL = r"https://cloud.flowiseai.com/api/v1/prediction/034ff02d-e9e0-4003-937d-6c37ea84e157"

In [4]:
with open(r"E:\Lenovo Ideapad 330\company-material\digital-workforce-transformation\ai-upskill-9\key-vault\nvd-database\api.key") as f:
    nvd_api_key = f.read().strip()
NVD_API_KEY = nvd_api_key

In [5]:
MODEL = "gpt-4.1-mini"
llm = ChatOpenAI(model=MODEL, temperature=0)

### 3. Build the Tools

#### CIS Tool

In [8]:
def query(payload):
    response = requests.post(CIS_URL, json=payload)
    return response.json()

output = query({
    "question": "Tell me about password policy in windows",
})

In [10]:
print(output["text"])

In Windows, a password policy is a set of rules that governs how passwords are created and managed, enhancing security across the system. Here are the typical components of a Windows password policy:

1. **Password Length**: Specifies the minimum and maximum length of passwords. A longer password is generally more secure.

2. **Complexity Requirements**: Passwords must include a mix of uppercase letters, lowercase letters, numbers, and special characters. This complexity helps to prevent easy guessing of passwords.

3. **Password History**: This setting prevents users from reusing recent passwords. For example, a user may be required to create a new password that has not been used in the last five password changes.

4. **Maximum Password Age**: Defines how long a password can be used before it must be changed. Regularly changing passwords reduces the risk of compromised accounts.

5. **Minimum Password Age**: This ensures that users cannot change their password immediately after settin

In [ ]:
@tool
def ask_cis(question: str) -> str:
    """
    Query the Flowise knowledge base for CIS benchmark for windows related queries and return the response.
    
    Args:
        question: User's question to ask the Flowise chatbot.
    """
    payload = {
        #"query": question
        "question": question
    }

    try:
        #response = requests.post(r"http://127.0.0.1:8000/ask", json=payload, timeout=30)
        response = requests.post(CIS_URL, json=payload, timeout=30)
        response.raise_for_status()

        data = response.json()

        # Most Flowise deployments return {"text": "..."}
        return data.get("text", str(data))

    except requests.RequestException as e:
        return f"Error communicating with Flowise: {e}"

In [43]:
ask_cis.invoke("Tell me about proper shutdown of windows servers")

'{\'answer\': "1. **Answer**: Proper shutdown of Windows servers is crucial to maintain system integrity and prevent data loss. It is recommended that only authorized users, such as Administrators and specific users, have the rights to shut down the system. This ensures that unauthorized users cannot cause a denial-of-service (DoS) condition. The shutdown process should be performed gracefully to allow the operating system and applications to close properly, which helps prevent corruption of data and system files.\\n\\n2. **Relevant CIS Controls**: \\n   - **CIS Control 6.8**: Define and Maintain Role-Based Access Control\\n   - **CIS Control 2.2.21**: Ensure \'Force shutdown from a remote system\' is set to \'Administrators\'\\n   - **CIS Control 2.2.37**: Ensure \'Shut down the system\' is set to \'Administrators, Users\'\\n\\n3. **Summary**: Proper shutdown procedures for Windows servers involve restricting shutdown permissions to trusted users, ensuring graceful shutdowns to protec

#### NVD Tool

In [16]:
@tool
def lookup_cve(keyword:str)->str:
    """Query the NVD API for CVEs matching a keyword."""
    print("[TOOL]CVE Tool Activated")
    url = "https://services.nvd.nist.gov/rest/json/cves/2.0"
    headers = {"apiKey": NVD_API_KEY} if NVD_API_KEY else {}
    params = {"keywordSearch": keyword, "resultsPerPage": 3}
    try:
        r = requests.get(url, params=params, headers=headers, timeout=20)
        r.raise_for_status()
        data = r.json()
        vulns = data.get("vulnerabilities", [])
        if not vulns:
            return "No CVEs found."
        lines = []
        for v in vulns:
            c = v["cve"]
            lines.append(f"{c['id']}: {c['descriptions'][0]['value'][:180]}")
        return "\n".join(lines)
    except Exception as e:
        return f"CVE lookup failed: {e}"

In [18]:
lookup_cve.invoke("SMB")

[TOOL]CVE Tool Activated


'CVE-1999-1387: Windows NT 4.0 SP2 allows remote attackers to cause a denial of service (crash), possibly via malformed inputs or packets, such as those generated by a Linux smbmount command that \nCVE-1999-0225: Windows NT 4.0 allows remote attackers to cause a denial of service via a malformed SMB logon request in which the actual data size does not match the specified size.\nCVE-1999-0495: A remote attacker can gain access to a file system using ..  (dot dot) when accessing SMB shares.'

#### IP Config Tool

In [19]:
import subprocess

@tool
def ipconfig_tool() -> str:
    """
    Returns the Windows network configuration using the ipconfig command.
    """
    try:
        result = subprocess.run(
            ["ipconfig"],
            capture_output=True,
            text=True,
            check=True,
            shell=True
        )

        return result.stdout

    except subprocess.CalledProcessError as e:
        return f"Error executing ipconfig:\n{e.stderr}"

In [ ]:
print(ipconfig_tool.invoke({}))

### 4. Agent Layer

In [21]:
planner = create_agent(
    model=llm,
    tools=[],
    system_prompt="You are a planner. Decide weather RAG and CVE lookup is required"
)

In [22]:
retrieval_agent = create_agent(
    model=llm,
    tools=[ask_cis],
    system_prompt="Use the CIS policy tool to retrieve CIS benchmark guidance. Use the provided tools mandatorily"
)

In [23]:
threat_agent = create_agent(
    model=llm,
    tools=[lookup_cve],
    system_prompt="Use CVE lookup tool when threat intelligence is needed. Use the provided tools mandatorily"
)

In [24]:
validator_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt="Validate the evidence and produce a concise final answer"
)

In [25]:
response = planner.invoke({ 'messages': 'what is the password requriement for windows?'})

In [26]:
response

{'messages': [HumanMessage(content='what is the password requriement for windows?', additional_kwargs={}, response_metadata={}, id='26571bb0-dc49-4703-8a88-d7c18c94c086'),
  AIMessage(content='For the query about password requirements for Windows, a CVE (Common Vulnerabilities and Exposures) lookup is not required since this is a general security policy question rather than a vulnerability or exploit inquiry. However, a RAG (Retrieval-Augmented Generation) lookup would be useful to provide the most accurate and up-to-date password policy information for Windows.\n\nDecision:\n- RAG lookup: Yes\n- CVE lookup: No', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 36, 'total_tokens': 123, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model

### 5. State

In [27]:
class CyberState(TypedDict):
    query: str
    plan: str
    rag: str
    cves: str
    final: str

### 6. Nodes

In [28]:
def planner_node(state: CyberState):
    r = planner.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"plan": str(r)}

In [29]:
def rag_node(state: CyberState):
    r = retrieval_agent.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"rag": str(r)}

In [30]:
def cve_node(state: CyberState):
    r = threat_agent.invoke({
        "messages":[
            {"role":"user", "content":state["query"]}
        ]
    })
    return {"cves": str(r)}

In [31]:
def validator_node(state: CyberState):

    prompt = f"""
User Query:
{state["query"]}

Plan:
{state.get("plan", "")}

RAG:
{state.get("rag", "")}

CVEs:
{state.get("cves", "")}

SCORE:
validation score

Produce the final validated response.
Also, give a score from 0 to 10
"""
    r = validator_agent.invoke({
        "messages":[
            {"role":"user", "content":prompt}
        ]
    })
    return {"final": r}

### 7. Workflow

In [32]:
graph = StateGraph(CyberState)

graph.add_node("planner", planner_node)
graph.add_node("rag", rag_node)
graph.add_node("cve", cve_node)
graph.add_node("validator", validator_node)

graph.add_edge(START, "planner")
graph.add_edge("planner", "rag")
graph.add_edge("rag", "cve")
graph.add_edge("cve", "validator")
graph.add_edge("validator", END)

In [33]:
app = graph.compile()

### 8. Execute

In [34]:
result = app.invoke({
    "query": "How can I harden Windows SMB services against ransomware?"
})

In [35]:
result

{'query': 'How can I harden Windows SMB services against ransomware?',
 'plan': "{'messages': [HumanMessage(content='How can I harden Windows SMB services against ransomware?', additional_kwargs={}, response_metadata={}, id='ca2241bf-bf9d-4cdb-9550-dd4a15e18b44'), AIMessage(content='RAG (Retrieval-Augmented Generation) and CVE (Common Vulnerabilities and Exposures) lookup would be useful here to provide the most up-to-date and specific security measures and known vulnerabilities related to Windows SMB services and ransomware.\\n\\nI will proceed with RAG and CVE lookup to gather detailed and current information on hardening Windows SMB services against ransomware.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 37, 'total_tokens': 110, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'c

In [36]:
print(result["final"]["messages"][-1].content)

To harden Windows SMB services against ransomware, follow these validated best practices aligned with CIS benchmarks and security advisories:

1. **Disable SMBv1**: SMBv1 is outdated and vulnerable. Disable it on all systems to eliminate known attack vectors.

2. **Enforce SMBv3 or higher**: Configure your environment to use SMB version 3.1.1 or at least SMBv2 by setting the minimum SMB dialect in the registry:
   - Path: `HKLM\SOFTWARE\Policies\Microsoft\Windows\LanmanServer:MinSmb2Dialect`
   - Value: `785` for SMBv3.1.1

3. **Disable sending unencrypted passwords**: Prevent plaintext password transmission by disabling the policy:
   - Group Policy Path: `Computer Configuration\Policies\Windows Settings\Security Settings\Local Policies\Security Options\Microsoft network client: Send unencrypted password to third-party SMB servers`
   - Set to Disabled

4. **Apply all security patches promptly**: Keep Windows systems updated to protect against known SMB vulnerabilities exploited by ra

In [37]:
result = app.invoke({
    "query": "How can I secure windows SMB against latest attacks?"
})

[TOOL]CVE Tool Activated


In [ ]:
result

In [39]:
print(result["final"]["messages"][-1].content)

To secure Windows SMB against the latest attacks, implement the following validated best practices:

1. **Disable SMBv1**: SMBv1 is outdated and vulnerable. Disable it and use SMBv2 or SMBv3, which offer improved security features.

2. **Enable SMB Signing**: This authenticates both client and server during communication, preventing man-in-the-middle attacks.

3. **Set Minimum SMB Version**: Configure Group Policy to enforce a minimum SMB version of SMBv2 or higher, ideally SMBv3.1.1, to avoid legacy protocol vulnerabilities.

4. **Use Strong Authentication**: Enforce strong, complex passwords and consider multi-factor authentication for SMB access.

5. **Restrict SMB Access**: Limit SMB access to only necessary users and devices. Use firewall rules and network segmentation to block SMB traffic from untrusted sources.

6. **Regular Auditing and Monitoring**: Continuously audit SMB share permissions and monitor network traffic for unusual activity to detect potential attacks early.

7. 